In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark_Postgres_Labb") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

jdbc_url = "jdbc:postgresql://postgres:5432/employee_db"
properties = {
    "user": "spark_user",
    "password": "spark_password",
    "driver": "org.postgresql.Driver"
}

In [4]:
df = spark.read.jdbc(url=jdbc_url, table="employees", properties=properties)

In [5]:
df.count()

25

In [7]:
df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- full_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: decimal(10,2) (nullable = true)
 |-- hire_date: date (nullable = true)



In [8]:
df.show(5)

+------+---------------+-----------+-------+----------+
|emp_id|      full_name| department| salary| hire_date|
+------+---------------+-----------+-------+----------+
|     1| Aydin Mammadov|Engineering|7200.00|2019-03-12|
|     2|  Leyla Aliyeva|Engineering|8400.00|2018-07-01|
|     3|   Rauf Hasanov|Engineering|4800.00|2022-11-15|
|     4| Nigar Quliyeva|Engineering|9500.00|2017-05-20|
|     5|Elvin Ismayilov|Engineering|5600.00|2021-02-08|
+------+---------------+-----------+-------+----------+
only showing top 5 rows



In [9]:
df_partitioned = spark.read.jdbc(
    url=jdbc_url, 
    table="employees", 
    column="emp_id", 
    lowerBound=1, 
    upperBound=25, 
    numPartitions=4, 
    properties=properties
)

In [11]:
pushdown_query = "(SELECT * FROM employees WHERE salary > 5000) AS filtered_data"

df_filtered = spark.read.jdbc(
    url=jdbc_url, 
    table=pushdown_query, 
    properties={**properties, "fetchsize": "1000"} # her defe 1000 setr melumat getirir
)

In [12]:
df_avg = df_filtered.groupBy("department").agg({"salary": "avg"})

In [13]:
df_avg.write.jdbc(
    url=jdbc_url, 
    table="dept_avg_salary", 
    mode="overwrite", 
    properties={**properties, "batchsize": "500"}
)

In [14]:
spark.read.jdbc(url=jdbc_url, table="dept_avg_salary", properties=properties).show()

+-----------+-----------+
| department|avg(salary)|
+-----------+-----------+
|      Sales|6300.000000|
|Engineering|7480.000000|
|         HR|5100.000000|
|    Finance|7100.000000|
|  Marketing|5966.666667|
+-----------+-----------+

